# SMART TIME — Gate 4G QLoRA Retrain + Evidence

Retrain on the canonical 130-example contract, evaluate the fixed 12-case suite, and refuse PASS unless the scorer itself passes >= 10/12.

In [ ]:
!rm -rf /kaggle/working/SMART-TIME-
!git clone -b feat/v3-next https://github.com/msm147852/SMART-TIME-.git /kaggle/working/SMART-TIME-
%cd /kaggle/working/SMART-TIME-
!git lfs install
!git lfs pull
!git rev-parse HEAD


In [ ]:
!nvidia-smi
!python - <<'PY'
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())
assert torch.cuda.is_available()
PY


In [ ]:
!pip install -q -r infra/smart-ai/training/requirements.txt
!python -m pip check
!python - <<'PY'
from pathlib import Path
import json
files=['backend/ai/training/smart-time-v2.jsonl','backend/ai/training/smart-time-grounded-v1.jsonl','backend/ai/training/smart-time-sft.jsonl']
counts=[len([x for x in Path(f).read_text(encoding='utf-8').splitlines() if x.strip()]) for f in files]
print(counts)
assert counts==[45,45,40], counts
assert sum(json.loads(x)['category']=='negative' for x in Path(files[-1]).read_text(encoding='utf-8').splitlines())==20
PY


In [ ]:
!rm -rf infra/smart-ai/training/output
!BASE_MODEL=Qwen/Qwen3-4B USE_QLORA=1 USE_LORA=1 EPOCHS=3 LORA_R=16 LORA_ALPHA=32 MAX_LENGTH=1024 BATCH_SIZE=1 GRADIENT_ACCUMULATION_STEPS=16 python infra/smart-ai/training/train_smart_ai.py 2>&1 | tee /kaggle/working/gate4g-train.log
!python - <<'PY'
import re
from pathlib import Path
s=Path('/kaggle/working/gate4g-train.log').read_text(encoding='utf-8')
m=re.findall(r"['\"]?eval_loss['\"]?\s*[:=]\s*([0-9.]+)",s)
assert m, 'eval_loss not found'
loss=float(m[-1]); print('eval_loss=',loss); assert loss < 1.6
assert Path('infra/smart-ai/training/output/adapter_model.safetensors').exists()
PY


In [ ]:
!MODEL=/kaggle/working/SMART-TIME-/infra/smart-ai/training/output BASE_MODEL=Qwen/Qwen3-4B EVAL_OUTPUT=/kaggle/working/gate4g-predictions.jsonl python infra/smart-ai/training/evaluate_smart_ai_v4g.py
!node scripts/score-smart-ai-eval-v4g.mjs /kaggle/working/gate4g-predictions.jsonl


In [ ]:
import json,subprocess
from pathlib import Path
r=subprocess.run(['node','scripts/score-smart-ai-eval-v4g.mjs','/kaggle/working/gate4g-predictions.jsonl'],capture_output=True,text=True)
print(r.stdout)
assert r.returncode==0
result=json.loads(r.stdout); assert result['passed']>=10 and result['pass_rate']>=0.80
head=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
out=Path(f'docs/gate-4g/{head}-gate4g-pass.jsonl'); out.parent.mkdir(parents=True,exist_ok=True)
out.write_text(Path('/kaggle/working/gate4g-predictions.jsonl').read_text(encoding='utf-8'),encoding='utf-8')
print(out)


In [ ]:
!git add docs/gate-4g
!git commit -m "test(ai): record Gate 4G PASS evidence"
!git push origin feat/v3-next


## PASS rule

PASS requires scorer exit code 0, >=10/12, and canonical tool names. No normalization hack. main, storageAdapter, and AI Gateway remain untouched.